## [内置中间件](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

#### 加载环境

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

#### Model fallback

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model='deepseek:deepseek-fake',
    middleware=[ModelFallbackMiddleware(
        "deepseek:deepseek-flash",
    )]
)

messages = [HumanMessage("你好")]
resp = agent.invoke({'messages': messages})

last_msg = resp["messages"][-1]
print(last_msg)
print(last_msg.response_metadata.get("model_name"))

#### LLM tool selector

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"

@tool
def get_news() -> str:
    """查询今日国内新闻概要"""
    return "新版 iphone 今日发布"

selector_model = init_chat_model(
    model='deepseek:deepseek-flash',
    # 关闭思考模型
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

agent = create_agent(
    model='deepseek:deepseek-v4-pro',
    tools=[get_weather, get_news],
    middleware=[LLMToolSelectorMiddleware(
        model=selector_model,
        max_tools=0,
        always_include=["get_weather"]
    )]
)

config: RunnableConfig = {"configurable": {"thread_id": "session01"}}
messages = [
    HumanMessage("北京今天天气如何，以及今天新闻概要"),
]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### Tool error

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware

@tool
def divide(a: float, b: float) -> float:
    """计算两个数相除。"""
    if b == 0:
        raise ValueError("除数不能为0")
    return a / b

def on_tool_error(error: Exception, request) -> str | None:
    if isinstance(error, ValueError):
        return (
            "计算失败：除数不能为0。"
            "请向用户说明无法计算，不要擅自修改除数。"
        )
    # 其他异常不处理，继续抛出
    return None

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[divide],
    middleware=[ToolErrorMiddleware(on_tool_error)]
)

messages = [HumanMessage("一除以零等于多少")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

##### Tool retry

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    raise TimeoutError()

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather],
    middleware=[
        ToolRetryMiddleware(
            max_retries=5,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=6.0,
            jitter=True,
            retry_on=(TimeoutError,),
        )
    ]
)

messages = [HumanMessage("今天北京天气如何")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

##### Model retry

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware

agent = create_agent(
    # 配置一个异常的模型
    model='deepseek:deepseek-fake',
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            initial_delay=1.0,
        )
    ]
)

messages = [HumanMessage("你好")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### LLM tool emulator

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents.middleware import LLMToolEmulator

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather],
    middleware=[LLMToolEmulator(
        model="deepseek:deepseek-flash"
    )],
)

messages = [HumanMessage("今天北京天气如何")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### File search

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import FilesystemFileSearchMiddleware

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather],
    middleware=[FilesystemFileSearchMiddleware(
        root_path="../",
        use_ripgrep=True,
    )],
)

messages = [HumanMessage("找到包含 get_weather 函数的 Jupyter 文件")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()

#### Shell tool

In [ ]:
from pathlib import Path
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ShellToolMiddleware, HostExecutionPolicy

workspace = Path.cwd()
print(Path.cwd())

agent = create_agent(
    model='deepseek:deepseek-flash',
    middleware=[ShellToolMiddleware(
        workspace_root=workspace,
        execution_policy=HostExecutionPolicy(
            command_timeout=10,
        )
    )]
)

messages = [HumanMessage("在工作目录下找到包含 get_weather 函数的 Jupyter 文件有哪些")]
resp = agent.invoke({'messages': messages})
for msg in resp["messages"]:
    msg.pretty_print()